In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/MajorPRojectDatasets-cropRecommendation-INDIA/cleaned_india_historic_dataset.csv')
df.head()

,State,District,Crop,Year,Season,Area,Area Units,Production,Production Units,Yield
0,Andaman and Nicobar Islands,NICOBARS,Arecanut,2001-02,Kharif,1254.0,Hectare,2061.0,Tonnes,1.643541
1,Andaman and Nicobar Islands,NICOBARS,Arecanut,2002-03,Whole Year,1258.0,Hectare,2083.0,Tonnes,1.655803
2,Andaman and Nicobar Islands,NICOBARS,Arecanut,2003-04,Whole Year,1261.0,Hectare,1525.0,Tonnes,1.209358
3,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,Arecanut,2001-02,Kharif,3100.0,Hectare,5239.0,Tonnes,1.690000
4,Andaman and Nicobar Islands,SOUTH ANDAMANS,Arecanut,2002-03,Whole Year,3105.0,Hectare,5267.0,Tonnes,1.696296


using CatBoostClassifier

In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Load Data
df = pd.read_csv('/content/drive/MyDrive/MajorPRojectDatasets-cropRecommendation-INDIA/cleaned_india_historic_dataset.csv')

# Keep only relevant features
df = df[["State", "District", "Season", "Crop"]]

# Drop missing rows
df.dropna(inplace=True)

# Inputs & target
X = df.drop(columns=["Crop"])
y = df["Crop"]

# Encode crop labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Identify categorical columns
cat_cols = ["State", "District", "Season"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# CatBoost Model
model = CatBoostClassifier(
    iterations=600,
    depth=10,
    learning_rate=0.08,
    loss_function='MultiClass',
    verbose=0
)
model.fit(X_train, y_train, cat_features=cat_cols)

# Evaluation
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
def predict_crop(state, district, season):
    sample = [[state, district, season]]
    probs = model.predict_proba(sample)[0]

    results = sorted(zip(le.classes_, probs), key=lambda x: x[1], reverse=True)
    return results
